# 🏋️ AI Gym Coach — Ingest & Index Knowledge Base

This notebook:
1. **Loads** the fitness FAQ knowledge base (`data/fitness_faq.json`)
2. **Explores** the data structure
3. **Ingests** documents into a `minsearch` index
4. **Tests** the search index with sample queries
5. **Saves** the index to disk for reuse

> Stack: `minsearch` (in-memory TF-IDF + keyword search) • `Python 3.12` • No external API needed for indexing

## 1. Setup & Imports

In [54]:
import json
import pickle
import os
from pathlib import Path
from pprint import pprint
from datetime import datetime

import minsearch

print(f"minsearch version : {minsearch.__version__}")
print(f"Working directory  : {Path.cwd()}")

minsearch version : 0.2.0
Working directory  : D:\


## 2. Configuration

In [55]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR   = Path(".").resolve()          # project root (run from project root)
DATA_DIR   = BASE_DIR / "data"
INDEX_DIR  = BASE_DIR / "index"
KB_FILE    = DATA_DIR / "fitness_faq.json"
INDEX_FILE = INDEX_DIR / "fitness_index.pkl"

INDEX_DIR.mkdir(exist_ok=True)            # create index/ if it doesn't exist

# ── Index field config ─────────────────────────────────────────────────────
# Fields that minsearch will tokenise and search over
TEXT_FIELDS    = ["question", "answer", "topic"]   # full-text search fields
KEYWORD_FIELDS = ["category", "id"]                 # exact-match filter fields

print(f"Knowledge base : {KB_FILE}")
print(f"Index output   : {INDEX_FILE}")
print(f"Text fields    : {TEXT_FIELDS}")
print(f"Keyword fields : {KEYWORD_FIELDS}")

Knowledge base : D:\data\fitness_faq.json
Index output   : D:\index\fitness_index.pkl
Text fields    : ['question', 'answer', 'topic']
Keyword fields : ['category', 'id']


## 3. Load Knowledge Base

In [56]:
def load_knowledge_base(filepath: Path) -> list[dict]:
    """Load and validate the fitness FAQ knowledge base from JSON."""
    if not filepath.exists():
        raise FileNotFoundError(f"Knowledge base not found: {filepath}")

    with open(filepath, encoding="utf-8") as f:
        documents = json.load(f)

    if not isinstance(documents, list):
        raise ValueError("Expected a JSON array at the top level.")

    # Flatten keywords list → space-joined string for text search
    for doc in documents:
        if isinstance(doc.get("keywords"), list):
            doc["keywords_text"] = " ".join(doc["keywords"])

    print(f"✅  Loaded {len(documents)} documents from {filepath.name}")
    return documents

KB_FILE = Path(r"D:\0.SUNRISE\DS\AI\AI_Gym_Coach\data\fitness_faq.json")
documents = load_knowledge_base(KB_FILE)

✅  Loaded 38 documents from fitness_faq.json


## 4. Explore the Data

In [57]:
# ── Basic stats ─────────────────────────────────────────────────────────────
from collections import Counter

categories = Counter(d["category"] for d in documents)
topics     = Counter(d["topic"]    for d in documents)

print("📊 Documents per category:")
for cat, count in categories.most_common():
    print(f"   {cat:<20} {count} docs")

print("\n📊 Documents per topic (top 10):")
for topic, count in topics.most_common(10):
    print(f"   {topic:<25} {count} docs")

📊 Documents per category:
   exercises            20 docs
   workout_plans        15 docs
   recovery             3 docs

📊 Documents per topic (top 10):
   chest                     2 docs
   legs                      2 docs
   arms                      2 docs
   back                      1 docs
   shoulders                 1 docs
   core                      1 docs
   glutes                    1 docs
   beginner                  1 docs
   intermediate              1 docs
   hypertrophy               1 docs


In [58]:
# ── Preview first document ───────────────────────────────────────────────────
print("Sample document:")
print("-" * 60)
sample = {k: v for k, v in documents[0].items() if k != "keywords_text"}
pprint(sample)

Sample document:
------------------------------------------------------------
{'answer': 'The barbell bench press is widely considered the best compound '
           'exercise for building overall chest mass. It recruits the '
           'pectoralis major, anterior deltoids, and triceps. Supplement it '
           'with incline dumbbell presses to target the upper chest and cable '
           'flyes for chest isolation and stretch.',
 'category': 'exercises',
 'id': 'ex_001',
 'keywords': ['chest', 'bench press', 'pectoralis', 'compound', 'mass'],
 'question': 'What is the best exercise for building chest mass?',
 'topic': 'chest'}


In [59]:
# ── Schema check ─────────────────────────────────────────────────────────────
REQUIRED_FIELDS = {"id", "category", "topic", "question", "answer", "keywords"}

missing = [
    doc["id"]
    for doc in documents
    if not REQUIRED_FIELDS.issubset(doc.keys())
]

if missing:
    print(f"⚠️  Documents with missing fields: {missing}")
else:
    print(f"✅  All {len(documents)} documents pass schema validation")

✅  All 38 documents pass schema validation


## 5. Ingest & Build the minsearch Index

`minsearch.Index` builds an in-memory TF-IDF index over the specified `text_fields`  
and supports exact-match filtering via `keyword_fields`.

In [60]:
def build_index(documents: list[dict]) -> minsearch.Index:
    """Create and fit a minsearch Index from the document list."""
    index = minsearch.Index(
        text_fields    = TEXT_FIELDS + ["keywords_text"],  # include flattened keywords
        keyword_fields = KEYWORD_FIELDS,
    )
    index.fit(documents)
    return index


print("⚙️  Building index...")
start = datetime.now()
index = build_index(documents)
elapsed = (datetime.now() - start).total_seconds()

print(f"✅  Index built in {elapsed:.3f}s")
print(f"   Indexed {len(documents)} documents")
print(f"   Text fields    : {index.text_fields}")
print(f"   Keyword fields : {index.keyword_fields}")

⚙️  Building index...
✅  Index built in 0.063s
   Indexed 38 documents
   Text fields    : ['question', 'answer', 'topic', 'keywords_text']
   Keyword fields : ['category', 'id']


## 6. Test the Index — Sample Searches

In [61]:
def search(query: str, category: str = None, n_results: int = 3) -> list[dict]:
    """
    Search the knowledge base.

    Parameters
    ----------
    query      : natural-language question
    category   : optional filter ("exercises", "workout_plans", "recovery")
    n_results  : number of results to return
    """
    boost = {
        "question"     : 3.0,   # questions are most semantically relevant
        "answer"       : 2.0,
        "keywords_text": 1.5,
        "topic"        : 1.0,
    }
    filter_dict = {"category": category} if category else {}

    results = index.search(
        query       = query,
        boost_dict  = boost,
        filter_dict = filter_dict,
        num_results = n_results,
    )
    return results


def display_results(query: str, results: list[dict]) -> None:
    """Pretty-print search results."""
    print(f"🔍 Query : '{query}'")
    print(f"   Found  : {len(results)} result(s)\n")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] [{r['category'].upper()} / {r['topic']}] {r['question']}")
        print(f"       → {r['answer'][:200]}..." if len(r['answer']) > 200 else f"       → {r['answer']}")
        print()

In [62]:
# ── Test 1: General query ────────────────────────────────────────────────────
q = "How do I build bigger legs?"
results = search(q, n_results=3)
display_results(q, results)

🔍 Query : 'How do I build bigger legs?'
   Found  : 3 result(s)

  [1] [EXERCISES / legs] What are the best exercises for bigger legs?
       → Squats are the king of leg exercises. Barbell back squats and front squats build overall leg mass. Add Romanian deadlifts for hamstrings, leg press for quad volume, and walking lunges for quad and glu...

  [2] [EXERCISES / arms] How do I build bigger triceps?
       → The triceps make up about 2/3 of upper arm size. Best exercises: close-grip bench press and weighted dips (mass builders), skull crushers (long head), tricep pushdowns (lateral head), and overhead tri...

  [3] [EXERCISES / legs] What is the proper squat form?
       → Squat form checklist: (1) Stand shoulder-width apart, toes slightly out. (2) Brace your core and maintain a neutral spine. (3) Initiate by breaking at the hips and knees simultaneously. (4) Descend un...



In [63]:
# ── Test 2: Category-filtered query ─────────────────────────────────────────
q = "best program for strength"
results = search(q, category="workout_plans", n_results=3)
display_results(q, results)

🔍 Query : 'best program for strength'
   Found  : 3 result(s)

  [1] [WORKOUT_PLANS / strength] How do I train specifically for strength?
       → Strength training focuses on the 1-5 rep range with 80-95% of your 1-rep max. Rest 3-5 minutes between sets. Key movements: squat, deadlift, bench press, overhead press. Periodization is crucial - cyc...

  [2] [WORKOUT_PLANS / intermediate] What is a good intermediate lifting program?
       → Once linear progression stalls, move to an intermediate program. Popular options: GZCLP (4-day upper/lower split), PHUL (Power Hypertrophy Upper Lower), or Texas Method (3-day volume/recovery/intensit...

  [3] [WORKOUT_PLANS / beginner] What is a good workout plan for beginners?
       → Beginners should follow a full-body 3-day program (e.g., Mon/Wed/Fri). A proven structure: Squat, Bench Press, Barbell Row - all done 3x5. Progress by adding 2.5-5kg each session (linear progression)....



In [64]:
# ── Test 3: Recovery query ───────────────────────────────────────────────────
q = "I am sore after training, what should I do?"
results = search(q, category="recovery", n_results=3)
display_results(q, results)

🔍 Query : 'I am sore after training, what should I do?'
   Found  : 2 result(s)

  [1] [RECOVERY / stretching] Should I stretch after workouts?
       → Post-workout static stretching is beneficial for flexibility and relaxation. Hold stretches for 30-60 seconds per muscle. Focus on the muscles you trained. Dynamic stretching is better pre-workout. St...

  [2] [RECOVERY / soreness] What is DOMS and how do I recover from it?
       → DOMS (Delayed Onset Muscle Soreness) peaks 24-72 hours after training, caused by microtears in muscle fibers. Recovery strategies: light movement (active recovery), adequate protein intake, sleep, hyd...



In [65]:
# ── Test 4: Beginner-focused query ───────────────────────────────────────────
q = "I am new to the gym, where do I start?"
results = search(q, n_results=3)
display_results(q, results)

🔍 Query : 'I am new to the gym, where do I start?'
   Found  : 3 result(s)

  [1] [EXERCISES / form] What are the most common gym mistakes beginners make?
       → Top beginner mistakes: (1) Ego lifting with poor form - injury risk. (2) Not tracking workouts. (3) Skipping legs. (4) Not eating enough protein. (5) Switching programs every 2-3 weeks (program hoppin...

  [2] [EXERCISES / pull_ups] How do I progress to my first pull-up?
       → Pull-up progression: (1) Dead hang (30+ seconds) to build grip and shoulder stability. (2) Negative pull-ups: jump to top, lower yourself slowly (3-5 sec). (3) Band-assisted pull-ups. (4) Lat pulldown...

  [3] [EXERCISES / cardio] Should I do cardio on leg day?
       → Doing intense cardio on leg day can interfere with recovery (concurrent training effect). If you must do cardio on leg day, do it AFTER lifting, keep it low-intensity (walking, light cycling), and lim...



In [66]:
# ── Test 5: Technique query ──────────────────────────────────────────────────
q = "correct form for squats"
results = search(q, category="exercises", n_results=3)
display_results(q, results)

🔍 Query : 'correct form for squats'
   Found  : 3 result(s)

  [1] [EXERCISES / chest] What is the correct bench press form?
       → Proper bench press form: (1) Lie flat, eyes under the bar. (2) Grip slightly wider than shoulder-width. (3) Retract and depress scapulae (create an arch). (4) Unrack and lower bar to lower chest (nipp...

  [2] [EXERCISES / legs] What are the best exercises for bigger legs?
       → Squats are the king of leg exercises. Barbell back squats and front squats build overall leg mass. Add Romanian deadlifts for hamstrings, leg press for quad volume, and walking lunges for quad and glu...

  [3] [EXERCISES / form] What are the most common gym mistakes beginners make?
       → Top beginner mistakes: (1) Ego lifting with poor form - injury risk. (2) Not tracking workouts. (3) Skipping legs. (4) Not eating enough protein. (5) Switching programs every 2-3 weeks (program hoppin...



## 7. Persist the Index to Disk

In [67]:
def save_index(index: minsearch.Index, path: Path) -> None:
    """Serialize the fitted index to disk using pickle."""
    with open(path, "wb") as f:
        pickle.dump(index, f)
    size_kb = path.stat().st_size / 1024
    print(f"💾  Index saved → {path}  ({size_kb:.1f} KB)")


def load_index(path: Path) -> minsearch.Index:
    """Load a previously serialized index from disk."""
    with open(path, "rb") as f:
        idx = pickle.load(f)
    print(f"📂  Index loaded ← {path}")
    return idx


save_index(index, INDEX_FILE)

💾  Index saved → D:\index\fitness_index.pkl  (71.6 KB)


In [68]:
# ── Verify round-trip (save → load → search) ─────────────────────────────────
index_reloaded = load_index(INDEX_FILE)

q = "how to warm up before lifting"
results = index_reloaded.search(
    query       = q,
    boost_dict  = {"question": 3, "answer": 2, "keywords_text": 1.5},
    num_results = 2,
)

print(f"🔄  Round-trip verify — query: '{q}'")
for r in results:
    print(f"   → [{r['category']}] {r['question']}")

📂  Index loaded ← D:\index\fitness_index.pkl
🔄  Round-trip verify — query: 'how to warm up before lifting'
   → [workout_plans] How should I warm up before lifting?
   → [exercises] How do I progress to my first pull-up?


## 8. Summary & Index Metadata

In [69]:
metadata = {
    "index_file"      : str(INDEX_FILE),
    "source_file"     : str(KB_FILE),
    "num_documents"   : len(documents),
    "categories"      : dict(categories),
    "text_fields"     : TEXT_FIELDS + ["keywords_text"],
    "keyword_fields"  : KEYWORD_FIELDS,
    "build_time_s"    : elapsed,
    "created_at"      : datetime.now().isoformat(),
}

# Save metadata alongside the index
meta_path = INDEX_DIR / "index_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("📋 Index Metadata")
print("-" * 50)
pprint(metadata)

📋 Index Metadata
--------------------------------------------------
{'build_time_s': 0.063234,
 'categories': {'exercises': 20, 'recovery': 3, 'workout_plans': 15},
 'created_at': '2026-10-17T05:53:41.685923',
 'index_file': 'D:\\index\\fitness_index.pkl',
 'keyword_fields': ['category', 'id'],
 'num_documents': 38,
 'source_file': 'D:\\0.SUNRISE\\DS\\AI\\AI_Gym_Coach\\data\\fitness_faq.json',
 'text_fields': ['question', 'answer', 'topic', 'keywords_text']}


---
## ✅ Ingest Complete!

| Item | Value |
|------|-------|
| Documents indexed | **40** |
| Categories | exercises · workout_plans · recovery |
| Index file | `index/fitness_index.pkl` |
| Metadata | `index/index_metadata.json` |

**Next step →** Use `load_index(INDEX_FILE)` in your RAG pipeline or chatbot to search the knowledge base.